# GSM-IC Reproduction — Irrelevant Context Distraction

**Community Benchmark** | Reproduction of Shi et al. (2023) "Large Language Models Can Be Easily Distracted by Irrelevant Context"

Tests whether LLMs can solve grade-school math problems when irrelevant information is added to the problem statement.

**Original paper:** arXiv:2302.00093

**Design:** Each problem has a clean version (no distraction) and a distracted version (with 1-3 irrelevant sentences added). The irrelevant sentences contain numbers and operations that could plausibly be part of the solution but are not.

**Metric:** Accuracy drop from clean to distracted conditions.

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
import numpy as np
import json
import re
import random

print(f"Available models: {list(kbench.llms.keys())}")

In [ ]:
def strip_thinking(text: str) -> str:
    if "</think>" in text:
        return text.split("</think>", 1)[1].strip()
    return text.strip()

def extract_number(response: str) -> float | None:
    """Extract the final numerical answer from a response."""
    response = strip_thinking(response)
    # Look for #### N pattern (GSM format)
    match = re.search(r'####\s*([\d,\.]+)', response)
    if match:
        return float(match.group(1).replace(',', ''))
    # Look for 'the answer is N' pattern
    match = re.search(r'(?:the answer is|answer:|final answer:?)\s*\$?([\d,\.]+)', response, re.IGNORECASE)
    if match:
        return float(match.group(1).replace(',', ''))
    # Last number in response
    numbers = re.findall(r'[\d,]+\.?\d*', response)
    if numbers:
        return float(numbers[-1].replace(',', ''))
    return None

In [ ]:
# Grade-school math problems with controlled distractors
# Each problem: clean statement, answer, and distractor sentences

PROBLEMS = [
    {
        "clean": "Sarah has 24 apples. She gives 8 apples to her friend Tom. How many apples does Sarah have now?",
        "answer": 16,
        "distractors": [
            "Tom also has 15 oranges in his backpack.",
            "The store sells apples for $3 each.",
            "Sarah's sister has 32 apples at home.",
        ],
    },
    {
        "clean": "A store has 150 shirts. They sell 45 shirts on Monday and 38 shirts on Tuesday. How many shirts are left?",
        "answer": 67,
        "distractors": [
            "The store also received 60 pairs of pants on Wednesday.",
            "Each shirt costs $25 before the discount.",
            "The store next door sold 92 hats last week.",
        ],
    },
    {
        "clean": "Maria reads 12 pages per hour. She reads for 5 hours. How many pages did she read in total?",
        "answer": 60,
        "distractors": [
            "Her brother reads 8 pages per hour.",
            "The book has 320 pages in total.",
            "Maria also spent 3 hours watching television.",
        ],
    },
    {
        "clean": "A farmer has 3 fields. Each field has 28 cows. How many cows does the farmer have in total?",
        "answer": 84,
        "distractors": [
            "The farmer also has 45 sheep in a separate barn.",
            "Each cow produces 6 gallons of milk per day.",
            "His neighbor has 5 fields with 20 cows each.",
        ],
    },
    {
        "clean": "A bus carries 42 passengers. At the first stop, 15 passengers get off and 8 get on. How many passengers are on the bus now?",
        "answer": 35,
        "distractors": [
            "The bus ticket costs $2.50 per ride.",
            "Another bus on a different route carries 55 passengers.",
            "The bus driver has been working for 12 hours today.",
        ],
    },
    {
        "clean": "A baker makes 96 cookies. She puts them equally into 8 boxes. How many cookies are in each box?",
        "answer": 12,
        "distractors": [
            "She also baked 48 cupcakes for a different order.",
            "Each cookie requires 15 grams of sugar.",
            "The bakery sold 200 items last Saturday.",
        ],
    },
    {
        "clean": "Tom saves $15 per week. After 8 weeks, how much money has he saved?",
        "answer": 120,
        "distractors": [
            "His sister saves $22 per week.",
            "Tom spent $35 on a new game last month.",
            "The bank offers 3% interest on savings accounts.",
        ],
    },
    {
        "clean": "A garden has 6 rows of flowers. Each row has 14 flowers. How many flowers are in the garden?",
        "answer": 84,
        "distractors": [
            "The garden also has 3 rows of vegetables with 20 plants each.",
            "Each flower needs 250 ml of water per day.",
            "The neighbor's garden has 120 flowers.",
        ],
    },
    {
        "clean": "A school has 480 students. One-third of them are in the science club. How many students are in the science club?",
        "answer": 160,
        "distractors": [
            "The school has 32 teachers.",
            "The math club has 95 members.",
            "Last year the school had 520 students.",
        ],
    },
    {
        "clean": "A train travels at 80 kilometers per hour for 3 hours. How far does the train travel?",
        "answer": 240,
        "distractors": [
            "The train has 12 carriages with 50 seats each.",
            "A faster express train travels at 120 km per hour.",
            "The ticket for the journey costs $45.",
        ],
    },
]

random.seed(42)
data = []
task_id = 0

for problem in PROBLEMS:
    # Clean version (no distraction)
    clean_prompt = (
        f"{problem['clean']}\n\n"
        "Solve step by step, then give your final numerical answer after ####."
    )
    data.append({
        "task_id": task_id,
        "prompt": clean_prompt,
        "answer": problem["answer"],
        "num_distractors": 0,
        "condition": "clean",
    })
    task_id += 1

    # Distracted versions (1, 2, 3 distractors)
    for n_dist in [1, 2, 3]:
        selected = problem["distractors"][:n_dist]
        # Insert distractors at random positions in the problem
        sentences = problem["clean"].split(". ")
        if len(sentences) >= 2:
            # Insert after first sentence
            modified = sentences[0] + ". " + " ".join(selected) + " " + ". ".join(sentences[1:])
        else:
            modified = " ".join(selected) + " " + problem["clean"]

        dist_prompt = (
            f"{modified}\n\n"
            "Solve step by step, then give your final numerical answer after ####."
        )
        data.append({
            "task_id": task_id,
            "prompt": dist_prompt,
            "answer": problem["answer"],
            "num_distractors": n_dist,
            "condition": f"distracted_{n_dist}",
        })
        task_id += 1

df_all = pd.DataFrame(data)
print(f"Generated {len(df_all)} items")
print(f"By condition: {dict(df_all['condition'].value_counts())}")

## Task Definition

In [ ]:
@kbench.task(
    name="gsm_ic_distraction",
    description="Grade-school math with irrelevant context — reproduction of Shi et al. (2023) GSM-IC"
)
def gsm_ic_distraction(
    llm,
    prompt: str,
    answer: float,
    num_distractors: int,
) -> bool:
    response = llm.prompt(prompt)
    extracted = extract_number(response)
    if extracted is None:
        return False
    return abs(extracted - float(answer)) < 0.01

## Run Evaluation

In [ ]:
eval_df = df_all[["prompt", "answer", "num_distractors"]].copy()

print(f"Running {len(eval_df)} items with kbench.llm...")
runs = gsm_ic_distraction.evaluate(
    llm=kbench.llm,
    evaluation_data=eval_df,
    n_jobs=2,
    timeout=120,
    max_attempts=2,
)
results = runs.as_dataframe()
results["correct"] = results["result"].apply(lambda r: float(r) if isinstance(r, bool) else float(r))
results = results.merge(df_all[["prompt", "condition", "num_distractors"]], on="prompt", how="left")
print(f"Collected {len(results)} results")

## Results & Analysis

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib"], check=True)
import matplotlib.pyplot as plt

print("=== Accuracy by Number of Distractors ===")
cond_stats = results.groupby("num_distractors")["correct"].agg(["mean", "count"])
print(cond_stats.to_string(float_format="%.3f"))

clean_acc = results[results["num_distractors"] == 0]["correct"].mean()
print(f"\nClean accuracy: {clean_acc:.2%}")
for n in [1, 2, 3]:
    dist_acc = results[results["num_distractors"] == n]["correct"].mean()
    print(f"With {n} distractor(s): {dist_acc:.2%} (drop: {clean_acc - dist_acc:+.2%})")

fig, ax = plt.subplots(figsize=(8, 5))
x = [0, 1, 2, 3]
y = [results[results["num_distractors"] == n]["correct"].mean() for n in x]
ax.plot(x, y, "o-", linewidth=2, markersize=10, color="steelblue")
ax.set_xlabel("Number of Distractor Sentences")
ax.set_ylabel("Accuracy")
ax.set_title("GSM-IC: Effect of Irrelevant Context on Math Problem Solving")
ax.set_xticks(x)
ax.set_ylim(0, 1.05)
ax.axhline(y=clean_acc, color="gray", linestyle="--", alpha=0.5, label=f"Clean baseline ({clean_acc:.0%})")
ax.legend()
plt.tight_layout()
plt.savefig("gsm_ic_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nPlot saved to gsm_ic_results.png")
print("\nReference: Shi et al. (2023) reported 20-40% accuracy drops with irrelevant context.")